In [1]:
import pandas as pd

In [4]:
df = pd.read_excel(r"C:\Users\HP\Desktop\Data Science\Transit Delay and Commute Predictor\ttc-bus-delay-data-2024.xlsx")

In [5]:
print(df.shape)
df.info()
df["Min Delay"].describe()

(59643, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59643 entries, 0 to 59642
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       59643 non-null  datetime64[ns]
 1   Route      59021 non-null  object        
 2   Time       59643 non-null  object        
 3   Day        59643 non-null  object        
 4   Location   59643 non-null  object        
 5   Incident   59643 non-null  object        
 6   Min Delay  59643 non-null  int64         
 7   Min Gap    59643 non-null  int64         
 8   Direction  50112 non-null  object        
 9   Vehicle    59643 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(6)
memory usage: 4.6+ MB


count    59643.000000
mean        21.218433
std         53.808012
min          0.000000
25%          8.000000
50%         11.000000
75%         20.000000
max        975.000000
Name: Min Delay, dtype: float64

In [9]:
print(df.isna().sum())                      # missing counts, should match 622 and 9531
print(df["Min Delay"].mean(), df["Min Delay"].median())
print((df["Min Delay"] == 0).sum())         # how many zero-delay rows?
print(df["Min Delay"].max() / 60)           # max delay in hours

Date            0
Route         622
Time            0
Day             0
Location        0
Incident        0
Min Delay       0
Min Gap         0
Direction    9531
Vehicle         0
dtype: int64
21.218433009741293 11.0
6733
16.25


In [10]:
# Look at Time first, to see its format
print(df["Time"].head())
print(df["Time"].apply(type).value_counts())


0    02:08
1    02:30
2    03:13
3    03:23
4    03:37
Name: Time, dtype: object
Time
<class 'str'>    59643
Name: count, dtype: int64


In [11]:
df["datetime"] = pd.to_datetime(
    df["Date"].dt.strftime("%Y-%m-%d") + " " + df["Time"].astype(str)
)
df["hour_ts"] = df["datetime"].dt.floor("h")   # rounded down to the hour, for the weather join later

In [12]:
print("Before:", len(df))
df = df.dropna(subset=["Route"])   # only 622 rows, safe to drop
df = df[df["Min Delay"] > 0]       # zero-minute delays aren't real delays
print("After:", len(df))

Before: 59643
After: 52815


In [13]:
print(df["Min Delay"].quantile([0.5, 0.9, 0.95, 0.99, 0.999]))

0.500     12.000
0.900     30.000
0.950     54.000
0.990    289.860
0.999    721.302
Name: Min Delay, dtype: float64
